<a href="https://colab.research.google.com/github/Max48732/-/blob/main/codelab6_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Технологии обработки больших данных

## Занятие 6. Машинное обучение в PySpark. Регрессия

1. Поставновка задачи регрессии и отличие от классификации
2. Линейная регрессия в pyspark
3. Метрики качества регрессии
4. Домашнее задание
  
**Рекомендованная литература:**
- Learning Spark 2 edition
- [Spark Regression](https://spark.apache.org/docs/latest/ml-classification-regression.html#regression)
- [Open Data Science Mlcourse ч.4](https://habr.com/ru/company/ods/blog/323890/)
- [Метрики качества регрессии](https://neerc.ifmo.ru/wiki/index.php?title=%D0%9E%D1%86%D0%B5%D0%BD%D0%BA%D0%B0_%D0%BA%D0%B0%D1%87%D0%B5%D1%81%D1%82%D0%B2%D0%B0_%D0%B2_%D0%B7%D0%B0%D0%B4%D0%B0%D1%87%D0%B0%D1%85_%D0%BA%D0%BB%D0%B0%D1%81%D1%81%D0%B8%D1%84%D0%B8%D0%BA%D0%B0%D1%86%D0%B8%D0%B8_%D0%B8_%D1%80%D0%B5%D0%B3%D1%80%D0%B5%D1%81%D1%81%D0%B8%D0%B8#.D0.9E.D1.86.D0.B5.D0.BD.D0.BA.D0.B8_.D0.BA.D0.B0.D1.87.D0.B5.D1.81.D1.82.D0.B2.D0.B0_.D1.80.D0.B5.D0.B3.D1.80.D0.B5.D1.81.D1.81.D0.B8.D0.B8)

## Установка pyspark в изолированой среде venv / conda env !

! pip install pyspark

In [ ]:
import pyspark

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

## 1. Поставновка задачи регрессии и отличие от классификации

![Classification](img/classification.png)

![sigmoid](img/sigmoid.jpg)

![Regression](img/regression.jpg)

Датасет - оценка медианной стоимости домовладения штата Калифорния в зависимости от характеристик квартала:
- longitude - Долгота положения центра квартала - чем меньше, тем западнее находится квартал (блок);
- latitude - Широта положения центра квартала - чем больше, тем севернее находится квартал (блок);
- housingMedianAge - средний возраст дома, чем больше, тем старше;
- totalRooms - Общее количество комнат в квартале;
- totalBedrooms - Общее количество спален в квартале;
- population - Общее количество людей, проживающих в пределах квартала;
- households - Общее число домашних хозяйств (группа людей или семья), проживающих в пределах квартала;
- medianIncome - Средний доход домашних хозяйств в пределах квартала (в десятках тысяч долларов США);
- medianHouseValue - Средняя стоимость дома в пределах квартала (в долларах США) - целевая переменная.

In [ ]:
data_path = 'sample_data/california_housing_test.csv'

In [ ]:
df = spark.read.format('csv').\
        options(header='true', inferschema='true').load(data_path,header=True)

df.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|  -122.05|   37.37|              27.0|     3885.0|         661.0|    1537.0|     606.0|       6.6085|          344700.0|
|   -118.3|   34.26|              43.0|     1510.0|         310.0|     809.0|     277.0|        3.599|          176500.0|
|  -117.81|   33.78|              27.0|     3589.0|         507.0|    1484.0|     495.0|       5.7934|          270500.0|
|  -118.36|   33.82|              28.0|       67.0|          15.0|      49.0|      11.0|       6.1359|          330000.0|
|  -119.67|   36.33|              19.0|     1241.0|         244.0|     850.0|     237.0|       2.9375|           81700.0|
+---------+--------+----

In [ ]:
df.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)



## 2. Линейная регрессия в pyspark

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler

In [ ]:
feature_vec = VectorAssembler(inputCols=['housing_median_age'], outputCol='Features')
vec_df = feature_vec.transform(df)

lr = LinearRegression(featuresCol='Features', labelCol='median_house_value', predictionCol='predict')

# Fit the model
lrModel = lr.fit(vec_df)

In [ ]:
# Print the coefficients and intercept for linear regression
print(f"median_house_value = {int(lrModel.coefficients.values)} * housing_median_age + {int(lrModel.intercept)}")

median_house_value = 823 * housing_median_age + 182090


/tmp/ipykernel_1447/738481565.py:2: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"median_house_value = {int(lrModel.coefficients.values)} * housing_median_age + {int(lrModel.intercept)}")


## 3. Метрики качества регрессии

$MSE = \dfrac{1}{n}\sum \limits_{i=1}^{n}(model(x_i) - y_i)^2$

MSE применяется в ситуациях, когда нам надо подчеркнуть большие ошибки и выбрать модель, которая дает меньше больших ошибок прогноза. Грубые ошибки становятся заметнее за счет того, что ошибку прогноза мы возводим в квадрат. И модель, которая дает нам меньшее значение среднеквадратической ошибки, можно сказать, что что у этой модели меньше грубых ошибок.

$RMSE = \sqrt{\dfrac{1}{n}\sum \limits_{i=1}^{n}(model(x_i) - y_i)^2}$

RMSE получается путем извлечения корня из MSE, в результате размерность ошибки в тех же величинах что и целевая переменная.

$MAE = \dfrac{1}{n}\sum \limits_{i=1}^{n}|model(x_i) - y_i|$

Среднеквадратичный функционал сильнее штрафует за большие отклонения по сравнению со среднеабсолютным, и поэтому более чувствителен к выбросам. При использовании любого из этих двух функционалов может быть полезно проанализировать, какие объекты вносят наибольший вклад в общую ошибку — не исключено, что на этих объектах была допущена ошибка при вычислении признаков или целевой величины.

Среднеквадратичная ошибка подходит для сравнения двух моделей или для контроля качества во время обучения, но не позволяет сделать выводов о том, на сколько хорошо данная модель решает задачу. Например, MSE = 10 является очень плохим показателем, если целевая переменная принимает значения от 0 до 1, и очень хорошим, если целевая переменная лежит в интервале (10000, 100000). В таких ситуациях вместо среднеквадратичной ошибки полезно использовать коэффициент детерминации — R2

$R^2 = 1 - \dfrac{\sum \limits_{i=1}^{n}(model(x_i) - y_i)^2}{\sum \limits_{i=1}^{n}(y_i - \overline{y})^2}$

Коэффициент детерминации измеряет долю дисперсии, объясненную моделью, в общей дисперсии целевой переменной. Фактически, данная мера качества — это нормированная среднеквадратичная ошибка. Если она близка к единице, то модель хорошо объясняет данные, если же она близка к нулю, то прогнозы сопоставимы по качеству с константным предсказанием.

In [ ]:
trainingSummary = lrModel.summary
trainingSummary.residuals.show()
print("RMSE: %f" % trainingSummary.rootMeanSquaredError)
print("r2: %f" % trainingSummary.r2)

+-------------------+
|          residuals|
+-------------------+
| 140373.47606918076|
| -41003.55632255983|
|  66173.47606918076|
| 124849.91154469695|
|-116038.00773494894|
|-145562.16917565712|
|-150503.55632255983|
|-30838.007734948944|
| -43.74963701382512|
| -43420.78202875439|
| -94150.68537152742|
|-154262.16917565712|
| 40861.395348826656|
| 35932.685838502395|
|  18173.47606918076|
| -63579.99179807605|
|-13143.749637013825|
| -66002.95940633546|
|  96497.04059366454|
| 203384.95987331046|
+-------------------+
only showing top 20 rows
RMSE: 112627.326567
r2: 0.008356


## 4. Домашнее задание

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import col, log, exp, when
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.master("local[*]").getOrCreate()

data_path = 'sample_data/california_housing_test.csv'
df = spark.read.format('csv').options(header='true', inferschema='true').load(data_path, header=True)

df = df.withColumn('log_house_value', log(col('median_house_value')))

df = df.withColumn('rooms_per_household', col('total_rooms') / col('households'))
df = df.withColumn('bedrooms_per_room', col('total_bedrooms') / col('total_rooms'))
df = df.withColumn('population_per_household', col('population') / col('households'))
df = df.withColumn('bedrooms_per_household', col('total_bedrooms') / col('households'))
df = df.withColumn('income_per_household', col('median_income') / col('households'))

df = df.withColumn('median_income_sq', col('median_income') ** 2)
df = df.withColumn('housing_median_age_sq', col('housing_median_age') ** 2)

df = df.withColumn('income_x_rooms', col('median_income') * col('total_rooms'))
df = df.withColumn('location_interaction', col('longitude') * col('latitude'))

df = df.replace([float('inf'), float('-inf')], [None, None])
df = df.na.drop()

feature_cols = [
    'longitude', 'latitude', 'housing_median_age',
    'total_rooms', 'total_bedrooms', 'population', 'households',
    'median_income',
    'rooms_per_household', 'bedrooms_per_room', 'population_per_household',
    'bedrooms_per_household', 'income_per_household',
    'median_income_sq', 'housing_median_age_sq',
    'income_x_rooms', 'location_interaction'
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol='raw_features')
df_assembled = assembler.transform(df)

scaler = StandardScaler(inputCol='raw_features', outputCol='features',
                       withStd=True, withMean=True)
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

train_df, test_df = df_scaled.randomSplit([0.8, 0.2], seed=42)

print("1. ЛИНЕЙНАЯ РЕГРЕССИЯ (предсказываем log цены)")

lr = LinearRegression(
    featuresCol='features',
    labelCol='log_house_value',
    predictionCol='log_prediction',
    maxIter=200,
    regParam=0.001,
    elasticNetParam=0.5
)

lr_model = lr.fit(train_df)
lr_pred = lr_model.transform(test_df)

lr_pred = lr_pred.withColumn('prediction', exp(col('log_prediction')))

evaluator = RegressionEvaluator(labelCol="median_house_value", predictionCol="prediction")
lr_r2 = evaluator.evaluate(lr_pred, {evaluator.metricName: 'r2'})
lr_rmse = evaluator.evaluate(lr_pred, {evaluator.metricName: 'rmse'})
print(f"Linear Regression R²: {lr_r2:.4f}")
print(f"Linear Regression RMSE: {lr_rmse:.2f}")

print("2. RANDOM FOREST")

rf = RandomForestRegressor(
    featuresCol='features',
    labelCol='median_house_value',
    predictionCol='prediction',
    numTrees=150,
    maxDepth=12,
    minInstancesPerNode=3,
    seed=42
)

rf_model = rf.fit(train_df)
rf_pred = rf_model.transform(test_df)

rf_r2 = evaluator.evaluate(rf_pred, {evaluator.metricName: 'r2'})
rf_rmse = evaluator.evaluate(rf_pred, {evaluator.metricName: 'rmse'})
print(f"Random Forest R²: {rf_r2:.4f}")
print(f"Random Forest RMSE: {rf_rmse:.2f}")

print("3. GRADIENT BOOSTED TREES")

gbt = GBTRegressor(
    featuresCol='features',
    labelCol='median_house_value',
    predictionCol='prediction',
    maxIter=100,
    maxDepth=8,
    stepSize=0.1,
    seed=42
)

gbt_model = gbt.fit(train_df)
gbt_pred = gbt_model.transform(test_df)

gbt_r2 = evaluator.evaluate(gbt_pred, {evaluator.metricName: 'r2'})
gbt_rmse = evaluator.evaluate(gbt_pred, {evaluator.metricName: 'rmse'})
print(f"Gradient Boosted Trees R²: {gbt_r2:.4f}")
print(f"Gradient Boosted Trees RMSE: {gbt_rmse:.2f}")

models = [
    ("Linear Regression (log)", lr_r2, lr_pred),
    ("Random Forest", rf_r2, rf_pred),
    ("Gradient Boosted Trees", gbt_r2, gbt_pred)
]

best_model = max(models, key=lambda x: x[1])
best_name, best_r2, best_pred = best_model

print(f"ЛУЧШАЯ МОДЕЛЬ: {best_name}")
print(f"R² = {best_r2:.4f}")


print("\nПримеры предсказаний:")
best_pred.select(
    col('median_house_value').alias('actual'),
    col('prediction').cast('int').alias('predicted'),
    ((col('prediction') - col('median_house_value')) / col('median_house_value') * 100).alias('error_%')
).show(15, truncate=False)

print("\nОшибки по диапазонам цен:")
best_pred.withColumn('price_range',
    when(col('median_house_value') < 100000, '< 100k')
    .when(col('median_house_value') < 200000, '100k-200k')
    .when(col('median_house_value') < 300000, '200k-300k')
    .when(col('median_house_value') < 400000, '300k-400k')
    .otherwise('> 400k')
).groupBy('price_range').agg(
    {'*': 'count',
     'median_house_value': 'avg',
     'prediction': 'avg'}
).withColumn('avg_error_%',
    (col('avg(prediction)') - col('avg(median_house_value)')) / col('avg(median_house_value)') * 100
).orderBy('price_range').show()


if best_name != "Linear Regression (log)":
    print("ТОП-10 ВАЖНЫХ ПРИЗНАКОВ:")


    if best_name == "Random Forest":
        importances = list(zip(feature_cols, rf_model.featureImportances))
    else:
        importances = list(zip(feature_cols, gbt_model.featureImportances))

    importances.sort(key=lambda x: x[1], reverse=True)
    for feature, importance in importances[:10]:
        print(f"{feature:25s}: {importance:.4f}")

if best_r2 > 0.6:
    print(f"УСПЕХ! R² = {best_r2:.4f} > 0.6")
    print(f"Модель {best_name} справилась с задачей!")
else:
    print(f"R² = {best_r2:.4f} < 0.6")

spark.stop()

1. ЛИНЕЙНАЯ РЕГРЕССИЯ (предсказываем log цены)
Linear Regression R²: 0.3877
Linear Regression RMSE: 88267.91
2. RANDOM FOREST
Random Forest R²: 0.6734
Random Forest RMSE: 64464.66
3. GRADIENT BOOSTED TREES
Gradient Boosted Trees R²: 0.5933
Gradient Boosted Trees RMSE: 71938.86
ЛУЧШАЯ МОДЕЛЬ: Random Forest
R² = 0.6734

Примеры предсказаний:
+--------+---------+-------------------+
|actual  |predicted|error_%            |
+--------+---------+-------------------+
|85600.0 |108840   |27.150116532606006 |
|104200.0|135500   |30.03908001718913  |
|100500.0|102310   |1.801294602821778  |
|102700.0|140195   |36.50938897619019  |
|136000.0|115711   |-14.917985369705372|
|79200.0 |100570   |26.982352707201123 |
|154600.0|146729   |-5.090669894496603 |
|37500.0 |87059    |132.15997203462769 |
|194400.0|186150   |-4.243347050911347 |
|75000.0 |85519    |14.025614010388846 |
|213800.0|155120   |-27.446093416858364|
|97300.0 |128334   |31.895276291363352 |
|197400.0|209220   |5.9880516634452245 |
|1

Добейтесь значения R2 > 0.6